<a href="https://colab.research.google.com/github/Denuwan392/Denuwan392/blob/main/background_remove.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kornia
!pip install fastapi uvicorn python-multipart pyngrok
!pip install fastapi[all]



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.1/442.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.2/168.2 kB 15.2 MB/s eta 0:00:00


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

print(torch.cuda.is_available())
model = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True)
torch.set_float32_matmul_precision(['high', 'highest'][0])
model.to('cuda')
model.eval()

# Data settings
image_size = (1024, 1024)
transform_image = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

from google.colab import files
uploaded = files.upload()
input_image_path = list(uploaded.keys())[0]  # Get the uploaded file path

image = Image.open(input_image_path)
input_images = transform_image(image).unsqueeze(0).to('cuda')

# Prediction
with torch.no_grad():
    preds = model(input_images)[-1].sigmoid().cpu()
pred = preds[0].squeeze()
pred_pil = transforms.ToPILImage()(pred)
mask = pred_pil.resize(image.size)
image.putalpha(mask)

# Save the result
output_path = "no_bg_image.png"
image.save(output_path)
print(f"Image saved to {output_path}")

# Download the imagess
files.download(output_path)


True


Saving concept-1868728_1920.jpg to concept-1868728_1920 (1).jpg
Image saved to no_bg_image.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import shutil
import torch
from PIL import Image
from io import BytesIO
from transformers import AutoModelForImageSegmentation
from pyngrok import ngrok
import uvicorn

# Initialize FastAPI app
app = FastAPI()

# Add CORS middleware to allow React app to communicate with the API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # For production, change this to the URL of your React app
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True)
model.to(device)
model.eval()

# Image settings
image_size = (1024, 1024)
transform_image = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# API endpoint to remove background
@app.post("/remove-background/")
async def remove_background(file: UploadFile = File(...)):
    with open("uploaded_image.png", "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    image = Image.open("uploaded_image.png")
    transformed_image = transform_image(image).unsqueeze(0).to(device)

    # Predict and process the image
    with torch.no_grad():
        preds = model(transformed_image)[-1].sigmoid().cpu()

    # Convert prediction to a mask
    pred = preds[0].squeeze()
    pred_pil = transforms.ToPILImage()(pred)
    mask = pred_pil.resize(image.size)

    # Apply mask to the original image
    image.putalpha(mask)

    # Save the output
    output_path = "no_bg_image.png"
    image.save(output_path)

    return FileResponse(output_path, media_type='image/png')

# Expose FastAPI app using Ngrok
public_url = ngrok.connect(8000)
print(f"FastAPI is accessible at {public_url}")

# Run the FastAPI app on port 8000
uvicorn.run(app, host="0.0.0.0", port=8000)


FastAPI is accessible at NgrokTunnel: "https://df64-34-16-216-147.ngrok-free.app" -> "http://localhost:8000"


INFO:     Started server process [595]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.247.48.69:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     103.247.48.69:0 - "GET / HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [595]


In [ ]:
!pip install pyngrok
!pip install uvicorn

In [ ]:
from pyngrok import ngrok
import uvicorn

# Set up ngrok for tunneling
ngrok.set_auth_token('2q8vBvHgsClNeNgolOOPMK0S4Nn_2iuGodgmXTWeC3eZ2fAc')
public_url = ngrok.connect(8000)
print(f"FastAPI is accessible at {public_url}")

# Run the FastAPI app
uvicorn.run(app, host="0.0.0.0", port=8000)


FastAPI is accessible at NgrokTunnel: "https://75f1-34-16-216-147.ngrok-free.app" -> "http://localhost:8000"


INFO:     Started server process [595]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [595]
